# ViT Optimizer Dynamics & Representation Geometry Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/colab/vit_optimizer_diagnostics/vit_optimizer_dynamics_geometry_lab.ipynb)

Small ViT 하나를 같은 초기값에서 여러 optimizer로 학습한 뒤, **학습 동역학 → 표현기하 → loss landscape → 함수공간**을 순서대로 비교한다.

기존 진단 코드는 그대로 재사용하고, 최근 논문 조사에서 비어 있던 다섯 축을 뒤에 추가한다.

1. empirical tangent-kernel spectrum / target alignment
2. input-output Jacobian singular spectrum
3. relative sharpness
4. neural-manifold axis geometry / empirical dichotomy capacity
5. parameter-trajectory PCA / path geometry

> 결과 출력은 한 셀에 몰아넣지 않는다. 각 진단량을 자기 셀에서 확인한다.

## 0. 설치와 저장소 준비

In [ ]:
!pip -q install datasets prodigyopt tensorboard scikit-learn

%cd /content
!rm -rf deep-learning-diagnostics-and-improvement
!git clone -q https://github.com/HisameOgasahara/deep-learning-diagnostics-and-improvement.git
%cd /content/deep-learning-diagnostics-and-improvement/colab/vit_optimizer_diagnostics

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from vit_lab_model_optim import SmallViT
from vit_lab_train import train_one_optimizer
from vit_lab_repr import (
    extract_features_and_logits,
    representation_diagnostics,
)
from vit_lab_landscape import (
    run_function_space_comparison,
    run_hessian_diagnostics,
    run_mode_connectivity,
)
from vit_lab_extended import (
    run_jacobian_diagnostics,
    run_manifold_diagnostics,
    run_relative_sharpness_diagnostics,
    run_tangent_kernel_diagnostics,
    run_trajectory_diagnostics,
)

## 1. 실험 설정

기본값은 Colab T4에서 반복 가능한 범위로 둔다. `EPOCHS=50`은 이전 100 epoch 실습보다 짧게 유지하고, 진단 checkpoint만 골라 저장한다.

In [ ]:
SEED = 42
EPOCHS = 50
DIAG_EPOCHS = [0, 1, 5, 10, 20, 35, 50]
DYNAMICS_EVERY = 20

TRAIN_SAMPLES = 10_000
VAL_SAMPLES = 2_000
BATCH_SIZE = 256
NUM_WORKERS = 4

OPTIMIZER_NAMES = ["sgd", "adamw", "prodigy", "muon"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

ROOT = Path("/content/vit_optimizer_diagnostics_outputs")
CKPT_DIR = ROOT / "checkpoints"
CSV_DIR = ROOT / "csv"
TB_DIR = ROOT / "tensorboard"

for directory in [CKPT_DIR, CSV_DIR, TB_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
print("outputs:", ROOT)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

## 2. CIFAR-10 준비

Hugging Face의 CIFAR-10을 한 번 받아 재사용한다. 학습용 augmentation loader와 표현 진단용 deterministic loader를 분리한다.

In [ ]:
class HFCifar10Dataset(Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        item = self.dataset[index]
        image = item["img"].convert("RGB")
        label = int(item["label"])
        return self.transform(image), label


mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

In [ ]:
raw = load_dataset("uoft-cs/cifar10")

train_split = raw["train"].shuffle(seed=SEED).select(range(TRAIN_SAMPLES))
val_split = raw["test"].shuffle(seed=SEED).select(range(VAL_SAMPLES))

train_dataset = HFCifar10Dataset(train_split, train_transform)
rep_train_dataset = HFCifar10Dataset(train_split, eval_transform)
val_dataset = HFCifar10Dataset(val_split, eval_transform)

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)

train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
rep_train_loader = DataLoader(rep_train_dataset, shuffle=False, **loader_kwargs)
rep_val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

print(len(train_dataset), len(val_dataset))

## 3. 같은 초기값 고정

In [ ]:
seed_everything(SEED)
initial_model = SmallViT().to(DEVICE)
initial_state = {
    name: tensor.detach().cpu().clone()
    for name, tensor in initial_model.state_dict().items()
}

init_train_features, _, init_train_labels = extract_features_and_logits(
    initial_model, rep_train_loader, DEVICE
)
init_val_features, init_val_logits, init_val_labels = extract_features_and_logits(
    initial_model, rep_val_loader, DEVICE
)

diagnostic_batch = next(iter(rep_val_loader))

del initial_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Optimizer별 학습

기존 `vit_lab_train.py`의 gradient / update / noise 기록 로직은 수정하지 않는다.

In [ ]:
history_frames = []
dynamics_frames = []
noise_frames = []

for optimizer_name in OPTIMIZER_NAMES:
    seed_everything(SEED)

    history_df, dynamics_df, noise_df = train_one_optimizer(
        run_name=optimizer_name,
        initial_state=initial_state,
        train_loader=train_loader,
        val_loader=rep_val_loader,
        device=DEVICE,
        epochs=EPOCHS,
        diag_epochs=DIAG_EPOCHS,
        dynamics_every=DYNAMICS_EVERY,
        ckpt_dir=CKPT_DIR,
        csv_dir=CSV_DIR,
        tb_dir=TB_DIR,
        amp_enabled=AMP_ENABLED,
    )

    history_frames.append(history_df)
    dynamics_frames.append(dynamics_df)
    noise_frames.append(noise_df)

history = pd.concat(history_frames, ignore_index=True)
dynamics = pd.concat(dynamics_frames, ignore_index=True)
gradient_noise = pd.concat(noise_frames, ignore_index=True)

### 4.1 Train / validation curve

In [ ]:
for run_name, frame in history.groupby("run"):
    plt.plot(frame["epoch"], frame["val_accuracy"], label=run_name)
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.legend()
plt.show()

history.groupby("run").tail(1)

### 4.2 Gradient norm

In [ ]:
gradient_norm = dynamics.groupby(["run", "parameter", "epoch"], as_index=False)["grad_norm"].mean()
gradient_norm.tail(20)

### 4.3 Consecutive gradient cosine

In [ ]:
gradient_cosine = dynamics.groupby(["run", "parameter", "epoch"], as_index=False)["grad_cosine_prev"].mean()
gradient_cosine.tail(20)

### 4.4 Update norm

In [ ]:
update_norm = dynamics.groupby(["run", "parameter", "epoch"], as_index=False)["update_norm"].mean()
update_norm.tail(20)

### 4.5 Update-to-weight ratio

In [ ]:
update_to_weight = dynamics.groupby(["run", "parameter", "epoch"], as_index=False)["update_to_weight"].mean()
update_to_weight.tail(20)

### 4.6 Consecutive update cosine

In [ ]:
update_cosine = dynamics.groupby(["run", "parameter", "epoch"], as_index=False)["update_cosine_prev"].mean()
update_cosine.tail(20)

### 4.7 Parameter displacement from initialization

In [ ]:
parameter_displacement = dynamics.groupby(["run", "parameter", "epoch"], as_index=False)["parameter_displacement"].mean()
parameter_displacement.tail(20)

### 4.8 Gradient noise ratio

In [ ]:
gradient_noise[["run", "epoch", "parameter", "grad_mean_norm", "grad_variance_trace", "gradient_noise_ratio"]].tail(20)

## 5. 기존 표현기하 진단 계산

기존 `vit_lab_repr.py`의 covariance spectrum, effective rank, CKA, linear probe, NC geometry를 그대로 호출한다.

In [ ]:
representation_df, spectrum_df, class_df = representation_diagnostics(
    optimizer_names=OPTIMIZER_NAMES,
    diag_epochs=DIAG_EPOCHS,
    initial_state=initial_state,
    init_train_features=init_train_features,
    init_train_labels=init_train_labels,
    init_val_features=init_val_features,
    init_val_logits=init_val_logits,
    init_val_labels=init_val_labels,
    rep_train_loader=rep_train_loader,
    rep_val_loader=rep_val_loader,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
)

print("representation diagnostics complete")

### 5.1 Covariance eigenspectrum

In [ ]:
spectrum_df.query("layer == 'penultimate' and rank <= 10").tail(40)

### 5.2 Effective rank

In [ ]:
representation_df[["run", "epoch", "layer", "effective_rank"]].tail(25)

### 5.3 CKA to initialization

In [ ]:
representation_df[["run", "epoch", "layer", "cka_to_init"]].tail(25)

### 5.4 Linear probe accuracy

In [ ]:
representation_df[["run", "epoch", "layer", "linear_probe_accuracy"]].tail(25)

### 5.5 NC1 — within / between class collapse

In [ ]:
class_df[["run", "epoch", "nc1_within_between"]]

### 5.6 NC2 — ETF error

In [ ]:
class_df[["run", "epoch", "nc2_etf_error"]]

### 5.7 NC3 — classifier / class-center alignment

In [ ]:
class_df[["run", "epoch", "nc3_classifier_alignment"]]

### 5.8 Classification margin

In [ ]:
class_df[["run", "epoch", "margin_mean", "margin_median", "margin_p10", "margin_positive_fraction"]]

### 5.9 k-NN purity

In [ ]:
class_df[["run", "epoch", "knn_purity"]]

### 5.10 Class radius / participation dimension / center correlation

In [ ]:
class_df[[
    "run",
    "epoch",
    "mean_class_radius",
    "mean_class_participation_dim",
    "class_center_abs_correlation",
]]

## 6. 기존 loss-landscape 진단

In [ ]:
hessian_df, hessian_summary = run_hessian_diagnostics(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    initial_state=initial_state,
    hessian_batch=diagnostic_batch,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
    steps=16,
)

print("Hessian diagnostics complete")

### 6.1 Hessian Ritz spectrum

In [ ]:
hessian_df[["run", "ritz_rank", "ritz_value"]]

### 6.2 Min / max curvature and spectral spread

In [ ]:
hessian_summary[["run", "min_ritz", "max_ritz", "spectral_spread"]]

### 6.3 Gradient alignment with top / minimum Hessian directions

In [ ]:
hessian_summary[["run", "grad_align_top", "grad_align_min"]]

## 7. 기존 함수공간 진단

In [ ]:
function_summary, function_pairwise = run_function_space_comparison(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    loader=rep_val_loader,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
)

print("function-space comparison complete")

### 7.1 Expected calibration error

In [ ]:
function_summary

### 7.2 Logit distance / prediction disagreement

In [ ]:
function_pairwise

## 8. 기존 mode connectivity

In [ ]:
connectivity_df, barrier_df = run_mode_connectivity(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    loader=rep_val_loader,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
    max_samples=1000,
)

print("mode connectivity complete")

### 8.1 Interpolation curve

In [ ]:
for pair, frame in connectivity_df.groupby(["run_a", "run_b"]):
    plt.plot(frame["alpha"], frame["val_loss"], marker="o", label=f"{pair[0]}-{pair[1]}")
plt.xlabel("interpolation alpha")
plt.ylabel("validation loss")
plt.legend()
plt.show()

### 8.2 Barrier height

In [ ]:
barrier_df

# 새로 추가한 5개 진단축

아래부터는 최근 2024–2026 학습동역학·표현기하 논문 조사에서 기존 노트북이 직접 커버하지 못했던 축이다. 계산량이 큰 양은 **무엇을 근사했는지 셀에 명시**한다.

## 9. Empirical tangent kernel — spectrum & target alignment

전체 multiclass NTK를 만드는 대신 Colab T4에서 반복할 수 있도록 **마지막 Transformer block + norm + head에 대한 true-class logit tangent features**를 사용한다. 따라서 결과 이름도 full NTK가 아니라 `partial empirical tangent kernel`로 해석한다.

In [ ]:
tangent_summary, tangent_spectrum = run_tangent_kernel_diagnostics(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    diagnostic_batch=diagnostic_batch,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
    sample_count=24,
)

print("tangent-kernel diagnostics complete")

### 9.1 Kernel-target alignment

In [ ]:
tangent_summary[["run", "kernel_target_alignment"]]

### 9.2 Tangent-kernel spectrum / effective rank

In [ ]:
display(tangent_summary[["run", "kernel_effective_rank", "kernel_trace", "kernel_top_eigenvalue"]])
display(tangent_spectrum.query("rank <= 10"))

## 10. Input-output Jacobian singular spectrum

Activation covariance는 **표현이 어디에 퍼져 있는가**를 보고, Jacobian은 **입력 방향 변화가 출력에서 얼마나 증폭·감쇠되는가**를 본다. 여기서는 각 이미지에서 `d logits / d input`의 singular values를 계산한다.

In [ ]:
jacobian_df, jacobian_summary = run_jacobian_diagnostics(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    diagnostic_batch=diagnostic_batch,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
    sample_count=8,
)

print("Jacobian diagnostics complete")

### 10.1 Jacobian spectral norm / Frobenius norm / participation rank

In [ ]:
jacobian_summary

### 10.2 Mean Jacobian singular-value spectrum

In [ ]:
jacobian_mean_spectrum = jacobian_df.groupby(["run", "rank"], as_index=False)["singular_value"].mean()

for run_name, frame in jacobian_mean_spectrum.groupby("run"):
    plt.plot(frame["rank"], frame["singular_value"], marker="o", label=run_name)
plt.xlabel("singular-value rank")
plt.ylabel("mean singular value")
plt.legend()
plt.show()

## 11. Relative sharpness

Raw Hessian eigenvalue는 parameter scaling에 민감하다. 여기서는 final classifier `W`에 대해 cross-entropy Hessian trace와 `||W||_F^2`를 결합한 layerwise relative sharpness를 계산한다.

이 SmallViT에서는 penultimate feature를 `h`, classifier를 `z = W h + b`로 두므로 Hessian trace를 정확히

`E[(1 - ||p||²) ||h||²]`

로 계산할 수 있어 full Hessian을 만들 필요가 없다.

In [ ]:
relative_sharpness_df = run_relative_sharpness_diagnostics(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    loader=rep_val_loader,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
    max_samples=512,
)

relative_sharpness_df

## 12. Neural-manifold axis geometry / empirical capacity

기존 class radius와 participation dimension에 **center-axis alignment**, **axis-axis alignment**를 추가한다.

`empirical_dichotomy_capacity`는 Chung 계열의 asymptotic replica-theory capacity 공식을 그대로 구현한 값이 아니다. CIFAR-10의 class manifold들에 random binary dichotomy를 부여한 뒤 한 linear hyperplane이 모든 point를 분리할 수 있었던 비율을 재는 **직접적인 empirical separability proxy**다.

In [ ]:
manifold_df = run_manifold_diagnostics(
    optimizer_names=OPTIMIZER_NAMES,
    final_epoch=EPOCHS,
    rep_val_loader=rep_val_loader,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
    device=DEVICE,
    max_samples=1000,
    dichotomy_trials=64,
)

print("manifold diagnostics complete")

### 12.1 Radius / dimension

In [ ]:
manifold_df[["run", "mean_class_radius", "mean_class_participation_dim"]]

### 12.2 Center-axis / axis-axis alignment

In [ ]:
manifold_df[["run", "mean_center_axis_alignment", "mean_axis_axis_alignment"]]

### 12.3 Empirical random-dichotomy capacity

In [ ]:
manifold_df[["run", "empirical_dichotomy_capacity"]]

## 13. Parameter-trajectory PCA / path geometry

Checkpoint를 각각 따로 보는 대신 `θ_0 → θ_1 → ... → θ_T` 전체를 하나의 경로로 본다. PCA는 실제 이동이 집중된 방향을 보여주고, path/chord ratio는 최종점까지 얼마나 우회했는지를 요약한다.

In [ ]:
trajectory_coordinates, trajectory_summary = run_trajectory_diagnostics(
    optimizer_names=OPTIMIZER_NAMES,
    diag_epochs=DIAG_EPOCHS,
    ckpt_dir=CKPT_DIR,
    csv_dir=CSV_DIR,
)

print("trajectory diagnostics complete")

### 13.1 Trajectory PCA coordinates

In [ ]:
for run_name, frame in trajectory_coordinates.groupby("run"):
    plt.plot(frame["pc1"], frame["pc2"], marker="o", label=run_name)
    for _, row in frame.iterrows():
        plt.annotate(str(int(row["epoch"])), (row["pc1"], row["pc2"]))
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

trajectory_coordinates

### 13.2 Explained variance / path geometry

In [ ]:
trajectory_summary

# 14. 새 진단축의 논문 근거

아래는 **노트북에 새로 추가한 부분만** 나중에 검증하기 쉽도록 기법별로 모아 둔 것이다. 기존 진단의 출처 목록과 섞지 않는다.

## 14.1 Empirical tangent kernel / target alignment

**Kumar, Bordelon, Gershman, Pehlevan — _Grokking as the Transition from Lazy to Rich Training Dynamics_, ICLR 2024.**
- 초기 NTK의 top eigendirections와 target의 alignment를 학습 regime 및 grokking과 연결한다.
- 공식 페이지: https://proceedings.iclr.cc/paper_files/paper/2024/hash/63ed15a46a143ff57484b38cd6b85d91-Abstract-Conference.html

**이 노트북의 구현 차이:** full multiclass NTK 대신 T4 비용을 제한하기 위해 마지막 block + norm + head에서 true-class logit gradient로 만든 partial empirical tangent kernel을 사용한다. 따라서 수치 자체를 논문의 full NTK 값과 동일시하면 안 된다.

## 14.2 Input-output Jacobian singular spectrum

**Li, Dai, Qu — _Understanding Generalizability of Diffusion Models Requires Rethinking the Hidden Gaussian Structure_, NeurIPS 2024.**
- denoiser input Jacobian의 leading singular vectors와 singular spectrum을 이용해 입력 방향이 출력에 어떻게 전달되는지 분석하고, leading singular-vector perturbation까지 수행한다.
- 공식 페이지: https://proceedings.neurips.cc/paper_files/paper/2024/hash/69e68611ec4d8c0ae4a4b2bece165f5f-Abstract-Conference.html

**이 노트북의 구현 차이:** diffusion denoiser가 아니라 ViT classifier의 `d logits / d image`를 사용한다. 핵심 수학 구조인 local Jacobian SVD를 ViT에 맞게 옮긴 것이다.

## 14.3 Relative sharpness

**Walter, Adilova, Vreeken, Kamp — _When Flatness Does (Not) Guarantee Adversarial Robustness_, ICLR 2026.**
- raw Hessian trace 대신 layer weight norm과 Hessian trace를 결합한 relative flatness / relative sharpness를 사용하고, penultimate representation과 input-space robustness를 연결한다.
- 공식 페이지: https://proceedings.iclr.cc/paper_files/paper/2026/hash/227277895277e1ec2422ae0c64f29c81-Abstract-Conference.html

**이 노트북의 구현:** `κ_Tr(W) = ||W||_F^2 Tr(H_W)`를 final classifier에서 계산한다. Cross-entropy + linear classifier 구조 덕분에 `Tr(H_W)`는 `E[(1-||p||²)||h||²]`로 정확히 계산한다.

## 14.4 Neural-manifold geometry / capacity

**Chou, Le, Wang, Chung — _Feature Learning beyond the Lazy-Rich Dichotomy: Insights from Representational Geometry_, ICML 2025 Spotlight.**
- class/task manifold의 radius, effective dimension, center/axis geometry, capacity 변화를 학습 중 추적해 representation untangling을 분석한다.
- 공식 페이지: https://proceedings.mlr.press/v267/chou25a.html

**이 노트북의 구현 차이:** radius와 participation dimension, center-axis/axis-axis alignment는 직접 계산하지만, 논문의 replica-theory manifold capacity를 축소 구현하지 않는다. 대신 random class dichotomy의 선형 완전분리 성공률을 `empirical_dichotomy_capacity`로 별도 표기한다.

## 14.5 Parameter trajectory / optimization-path geometry

**Guille-Escuret, Naganuma, Fatras, Mitliagkas — _No Wrong Turns: The Simple Geometry Of Neural Networks Optimization Paths_, ICML 2024.**
- 개별 checkpoint가 아니라 optimization path 위의 gradient geometry를 연구하고 RSI, error bound, 그 비율 γ를 시간축으로 추적한다.
- 공식 페이지: https://proceedings.mlr.press/v235/guille-escuret24a.html

**Song, Ahn, Yun — _Does SGD Really Happen in Tiny Subspaces?_, ICLR 2025.**
- 실제 SGD update와 Hessian dominant subspace의 alignment를 추적하고 projection intervention으로 관찰적 alignment의 기능적 의미를 검증한다.
- 공식 페이지: https://proceedings.iclr.cc/paper_files/paper/2025/hash/1757af1fe1429801bdf3abf5600f8bba-Abstract-Conference.html

**이 노트북의 구현 차이:** 저장된 checkpoint trajectory를 PCA로 투영하고 path length, endpoint chord, path/chord ratio, consecutive-step cosine을 계산한다. `No Wrong Turns`의 RSI/EB/γ를 그대로 재현하는 셀은 아니므로 결과 이름을 그 용어로 표기하지 않는다.

---

## 해석 원칙

- `covariance spectrum`과 `Jacobian spectrum`을 같은 것으로 해석하지 않는다.
- `partial tangent kernel`을 full NTK라고 부르지 않는다.
- `empirical_dichotomy_capacity`를 이론적 manifold capacity와 동일시하지 않는다.
- relative sharpness는 raw Hessian max eigenvalue를 대체하는 값이 아니라 **parameter scaling을 통제하는 보완 진단**으로 같이 본다.
- trajectory PCA는 많이 이동한 방향을 보여주고 Hessian eigenspace는 많이 휘는 방향을 보여주므로 둘을 구분한다.

# 15. 출력 파일

모든 수치는 `/content/vit_optimizer_diagnostics_outputs/csv`에 CSV로 남는다. 새 파일은 다음과 같다.

- `tangent_kernel_summary.csv` / `tangent_kernel_spectrum.csv`
- `jacobian_summary.csv` / `jacobian_spectrum.csv`
- `relative_sharpness.csv`
- `manifold_geometry_extended.csv`
- `trajectory_pca.csv` / `trajectory_path_summary.csv`